[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/data%20cleaning/practice/data_cleaning_worksheet.ipynb)

# Practice · data cleaning

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Ten questions covering Lectures 1 to 3: the missingness patterns, the three tests for the mechanism,
the sensitivity analysis and the imputation scoreboard. Each question names a variable. Put your
result in that variable and run the cell. The worked answer sits under **Answer**. Click it open
once you have tried.

**The data.** `panel.csv`, 500 households, written for this worksheet the way `survey.csv` was
written for the lectures: the mechanism is known because it was planted.

| column | mechanism | missing |
|---|---|---|
| `power_backup` | MCAR, a flat tablet-failure rate | 22 |
| `rent` | MAR on `age` and `household_size` | 154 |
| `savings` | MNAR on itself, low savers withhold | 74 |

`age`, `household_size`, `education_years` and `commute_min` are complete.

`truth_rent` and `truth_savings` hold the values before anything was deleted. No real study has
them, which is why the last two questions can be scored at all.


In [ ]:
import io
import requests
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from sklearn.impute import KNNImputer

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/data%20cleaning/practice/data/"
panel = pd.read_csv(URL + "panel.csv")

def load_npy(name):
    return np.load(io.BytesIO(requests.get(URL + name).content))

truth_rent = load_npy("truth_rent.npy")
truth_savings = load_npy("truth_savings.npy")

NUM = ["age", "household_size", "education_years", "commute_min", "rent", "savings"]
print(panel.isna().sum().to_string())

### Given: the ML mean and covariance under missingness

Little's test needs μ̂ and Σ̂ estimated from all 500 rows, which is the EM step from Lecture 2. It is
supplied here so Q5 is the assembly of the statistic rather than the estimator. Q1 to Q4 do not
need it.

In [ ]:
def em_mean_cov(X, iters=300, tol=1e-8):
    """ML mean and covariance of X (n by p, holding NaN), by EM."""
    X = np.asarray(X, float)
    n, p = X.shape
    mu = np.nanmean(X, 0)
    S = np.cov(np.where(np.isnan(X), mu, X), rowvar=False)
    for _ in range(iters):
        T1 = np.zeros(p)
        T2 = np.zeros((p, p))
        for i in range(n):
            row = X[i]
            obs = ~np.isnan(row)
            mis = ~obs
            z = row.copy()
            C = np.zeros((p, p))
            if mis.any():
                B = S[np.ix_(mis, obs)] @ np.linalg.pinv(S[np.ix_(obs, obs)])
                z[mis] = mu[mis] + B @ (row[obs] - mu[obs])
                C[np.ix_(mis, mis)] = S[np.ix_(mis, mis)] - B @ S[np.ix_(obs, mis)]
            T1 += z
            T2 += np.outer(z, z) + C
        mu_new, S_new = T1 / n, T2 / n - np.outer(T1 / n, T1 / n)
        done = max(np.abs(mu_new - mu).max(), np.abs(S_new - S).max()) < tol
        mu, S = mu_new, S_new
        if done:
            break
    return mu, S

### Q1  Missingness patterns

Count the rows of each missingness pattern over `rent`, `savings` and `power_backup`.
Seven of the eight possible patterns occur.

Answer variable `q1`: a Series indexed by the three booleans.

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
q1 = panel[["rent", "savings", "power_backup"]].isna().value_counts()
q1
```

```
rent   savings  power_backup
False  False    False           281
True   False    False           126
False  True     False            52
True   True     False            19
False  False    True             10
True   False    True              9
False  True     True              3
Name: count, dtype: int64
```

</details>

### Q2  Group-mean comparison on rent

Test 1 from Lecture 2. Split `age` by whether `rent` is missing and run Welch's t-test
across the two groups. Report the statistic and the p-value.

Answer variable `q2`: a tuple `(t, p)`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
R = panel["rent"].isna()
t, p = stats.ttest_ind(panel["age"][R], panel["age"][~R], equal_var=False)

q2 = (round(float(t), 3), float(f"{p:.3g}"))
q2
```

```
(3.992, 8.27e-05)
```

</details>

### Q3  The permutation version, on power_backup

The same comparison for `power_backup`, with the null built by relabelling instead of
looked up in a t distribution. 5,000 permutations, `np.random.default_rng(0)`, the statistic being
the difference in mean `age` between the two groups. Report the observed difference in years and
the two-sided p. A loop that draws differently will move the last digit.

Answer variable `q3`: a tuple `(observed_difference, p)`.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
rng = np.random.default_rng(0)
age = panel["age"].to_numpy()
B = panel["power_backup"].isna()
obs = age[B].mean() - age[~B].mean()

null = np.empty(5000)
for i in range(5000):
    idx = rng.permutation(len(panel))
    null[i] = age[idx[:B.sum()]].mean() - age[idx[B.sum():]].mean()

q3 = (round(float(obs), 3), round(float((np.abs(null) >= abs(obs)).mean()), 4))
q3
```

```
(2.762, 0.3578)
```

</details>

### Q4  Degrees of freedom for Little's test

Over the six columns in `NUM`, count the missingness patterns, the observed-column slots
they contribute between them, and the degrees of freedom `sum(|J_k|) - p`.

Answer variable `q4`: a tuple `(patterns, slots, df)`.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
patt = panel[NUM].notna().apply(tuple, axis=1)

slots = sum(sum(k) for k in patt.unique())
q4 = (len(patt.unique()), slots, slots - len(NUM))
q4
```

```
(4, 20, 14)
```

</details>

### Q5  Little's test on the six numeric columns

Assemble `d2 = sum_k n_k (xbar_k - mu_Jk)' inv(Sigma_Jk) (xbar_k - mu_Jk)`, taking μ̂ and Σ̂
from `em_mean_cov` on all 500 rows and cutting each down to the columns its pattern observes.
Report the statistic, its degrees of freedom and the p-value.

Answer variable `q5`: a tuple `(d2, df, p)`.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
def littles_test(df, cols):
    X = df[cols].to_numpy(float)
    mu, S = em_mean_cov(X)

    groups = {}
    for i, row in enumerate(~np.isnan(X)):
        groups.setdefault(tuple(row), []).append(i)

    d2, dfree = 0.0, 0
    for key, idx in groups.items():
        J = np.array(key, bool)
        if not J.any():
            continue
        diff = X[np.ix_(idx, np.where(J)[0])].mean(0) - mu[J]
        d2 += len(idx) * diff @ np.linalg.pinv(S[np.ix_(J, J)]) @ diff
        dfree += J.sum()
    dfree -= len(cols)
    return d2, int(dfree), float(stats.chi2.sf(d2, dfree))


d2, dfree, pval = littles_test(panel, NUM)
q5 = (round(float(d2), 2), dfree, float(f"{pval:.3g}"))
q5
```

```
(108.44, 14, 1.12e-16)
```

</details>

### Q6  Logistic on the rent missingness indicator

Test 3. Fit `rent.isna() ~ age + household_size + education_years` with `sm.Logit` and
report the model-level likelihood-ratio test: `2 * (llf - llnull)`, its degrees of freedom and its
p-value.

Answer variable `q6`: a tuple `(lr_chi2, df, p)`.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
PRED = ["age", "household_size", "education_years"]

def fit_logit(col):
    y = panel[col].isna().astype(int)
    return sm.Logit(y, sm.add_constant(panel[PRED])).fit(disp=0)


m = fit_logit("rent")
q6 = (round(float(2 * (m.llf - m.llnull)), 2), int(m.df_model),
      float(f"{m.llr_pvalue:.3g}"))
q6
```

```
(20.79, 3, 0.000116)
```

</details>

### Q7  Wald table for that fit

Report coefficient, standard error, odds ratio and Wald p for every term of the same fit,
intercept included. The odds ratio is `exp(coef)`, the multiplier on the odds of withholding per
unit of the predictor.

Answer variable `q7`: a DataFrame, 4 rows by 4 columns.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
q7 = pd.DataFrame({"coef": m.params.round(4), "se": m.bse.round(4),
                   "OR": np.exp(m.params).round(3),
                   "p": m.pvalues.map(lambda v: float(f"{v:.3g}"))})
q7
```

```
                   coef      se     OR         p
const           -2.2979  0.6103  0.100  0.000166
age              0.0329  0.0088  1.033  0.000192
household_size   0.1359  0.0623  1.146  0.029200
education_years -0.0397  0.0488  0.961  0.416000
```

</details>

### Q8  The same test on all three incomplete columns

Fit that model for `rent`, `savings` and `power_backup`, and report the LR p-value for each.
Two of the three reject. The test names a predictor of missingness, and separating MAR from MNAR
takes Q9.

Answer variable `q8`: a Series indexed by column.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
q8 = pd.Series({c: float(f"{fit_logit(c).llr_pvalue:.3g}")
                for c in ["rent", "savings", "power_backup"]}, name="LR p")
q8
```

```
rent            1.160000e-04
savings         2.870000e-21
power_backup    1.280000e-01
Name: LR p, dtype: float64
```

</details>

### Q9  Pattern mixture on savings

`savings` is the MNAR column, so `mu = mu_obs + pi * delta` from Lecture 2 applies. Report
the missing fraction π, the observed mean, the estimate at an assumed shift of δ = −300000, and the
true δ, which `truth_savings` can give and no real study can.

Answer variable `q9`: a tuple of four numbers.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
M = panel["savings"].isna()
pi = float(M.mean())
mu_obs = float(panel["savings"][~M].mean())
delta = -300000.0

q9 = (round(pi, 3), round(mu_obs, 0), round(mu_obs + pi * delta, 0),
      round(float(truth_savings[M].mean() - truth_savings[~M].mean()), 0))
q9
```

```
(0.148, 448432.0, 404032.0, -352310.0)
```

</details>

### Q10  Imputation scored against the truth

Fill `rent` four ways: the column mean; a regression on `age`, `household_size`,
`education_years` and `commute_min`; that regression plus a draw from `N(0, residual sd)` with
`np.random.default_rng(0)`; and `KNNImputer(n_neighbors=5)` over those four columns and `rent`.
Score each by RMSE against `truth_rent`, on the missing rows only.

Answer variable `q10`: a Series of four RMSEs.

In [ ]:
q10 = ...   # your answer
q10

<details>
<summary><b>Answer</b></summary>

```python
Rm = panel["rent"].isna().to_numpy()
XCOLS = ["age", "household_size", "education_years", "commute_min"]

def rmse(filled):
    return float(np.sqrt(np.mean((filled[Rm] - truth_rent[Rm]) ** 2)))


scores = {"mean": rmse(panel["rent"].fillna(panel["rent"].mean()).to_numpy())}

ols = sm.OLS(panel["rent"][~Rm], sm.add_constant(panel[XCOLS][~Rm])).fit()
pred = ols.predict(sm.add_constant(panel[XCOLS])).to_numpy()
scores["regression"] = rmse(np.where(Rm, pred, panel["rent"]))

rng2 = np.random.default_rng(0)
sd = float(np.sqrt(ols.mse_resid))
scores["stochastic regression"] = rmse(
    np.where(Rm, pred + rng2.normal(0, sd, len(panel)), panel["rent"]))

knn = KNNImputer(n_neighbors=5).fit_transform(panel[XCOLS + ["rent"]])
scores["knn k=5"] = rmse(knn[:, -1])

q10 = pd.Series(scores, name="rmse").round(0)
q10
```

```
mean                     20172.0
regression               13515.0
stochastic regression    17340.0
knn k=5                  15749.0
Name: rmse, dtype: float64
```

</details>